### Load Data

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os
from pathlib import Path

df = pd.read_csv("../data/raw/labeled_insurance.csv")
df.head()

,age,tenure,vehicle_type,claims_history,claims_count
0,56,10,Sedan,0,0
1,69,16,Sedan,0,0
2,46,12,Truck,0,0
3,32,0,SUV,1,1
4,60,1,Truck,1,1


###### age → policy holder age
###### tenure → years with the insurer
###### vehicle_type → categorical feature (SUV, Truck, etc.)
###### claims_history → past claims indicator
###### claims_count → number of claims

### Inspect

In [3]:
df.shape

(1000, 5)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             1000 non-null   int64 
 1   tenure          1000 non-null   int64 
 2   vehicle_type    1000 non-null   object
 3   claims_history  1000 non-null   int64 
 4   claims_count    1000 non-null   int64 
dtypes: int64(4), object(1)
memory usage: 39.2+ KB


In [5]:
df.describe()

,age,tenure,claims_history,claims_count
count,1000.000000,1000.000000,1000.000000,1000.000000
mean,49.857000,9.344000,0.521000,0.717000
std,18.114267,5.763061,0.688501,0.813371
min,18.000000,0.000000,0.000000,0.000000
25%,35.000000,4.000000,0.000000,0.000000
50%,50.000000,9.000000,0.000000,1.000000
75%,66.000000,14.000000,1.000000,1.000000
max,79.000000,19.000000,4.000000,4.000000


In [6]:
df.isna().sum()

age               0
tenure            0
vehicle_type      0
claims_history    0
claims_count      0
dtype: int64

### Column Efficacy

In [7]:
df.columns

Index(['age', 'tenure', 'vehicle_type', 'claims_history', 'claims_count'], dtype='object')

In [8]:
df.nunique().sort_values()

vehicle_type       4
claims_history     5
claims_count       5
tenure            20
age               62
dtype: int64

###### If a unique values were 1 or 1000(max value). Then we wouldn't be able to gain any knowledge from using that variable. Therefore, we would be able to remove that variable to reduce computation efforts.

### Seperating Numeric and Categorical Features

In [9]:
numeric_features = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = df.select_dtypes(include=["object", "category"]).columns.tolist()

In [10]:
print(numeric_features)

['age', 'tenure', 'claims_history', 'claims_count']


In [11]:
print(categorical_features)

['vehicle_type']


### Splitting Labeled data into train/test split

In [12]:
X = df.drop(columns=['claims_count'])
y = df['claims_count'] 

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [14]:
train_df = pd.concat([X_train, y_train], axis=1)
test_df = pd.concat([X_test, y_test], axis=1)

In [ ]:
PROJECT_ROOT = Path.cwd().parent
processed_dir = PROJECT_ROOT / "data" / "processed"

train_processed_dir = processed_dir / "new_train.csv"
test_processed_dir = processed_dir / "newtest.csv"

print(train_processed_dir)
print(train_df)

train_df.to_csv(str(train_processed_dir), index=False)
test_df.to_csv(str(test_processed_dir), index=False)

### Feature Engineering

###### Currently the age variable is numerical, making the data sparse. To make it dense, I will be turning it categorical.

In [37]:
# Converting age into buckets
bins = [17, 34, 49, 65, 80]
labels = ["Young", "Early_Middle", "Late_Middle", "Senior"]

train_df["age_bucket"] = pd.cut(train_df["age"], bins=bins, labels=labels)
test_df["age_bucket"] = pd.cut(test_df["age"], bins=bins, labels=labels)


KeyError: 'age'

In [ ]:
# reordering the cols
cols = ["age_bucket"] + [c for c in train_df.columns if c not in ["age", "age_bucket"]]
train_df = train_df[cols]

cols2 = ["age_bucket"] + [c for c in test_df.columns if c not in ["age", "age_bucket"]]
test_df = test_df[cols2]

print(test_df.head())

       age_bucket  tenure vehicle_type  claims_history  claims_count
521  Early_Middle       5        Sedan               1             2
737   Late_Middle      14        Sedan               0             0
740        Senior      13        Truck               0             0
660         Young       9        Truck               0             1
411        Senior      19        Truck               0             0


###### Applying onehot encoding to categorical variables

In [ ]:
train_df_encoded = pd.get_dummies(train_df, columns=['age_bucket', 'vehicle_type'], dtype=int)
test_df_encoded = pd.get_dummies(test_df, columns=['age_bucket', 'vehicle_type'], dtype=int)

###### Seperating X and y

In [ ]:
X_train = train_df_encoded.drop(columns=["claims_count"])
y_train = train_df_encoded["claims_count"]

X_test = test_df_encoded.drop(columns=["claims_count"])
y_test = test_df_encoded["claims_count"]